In [37]:
import pandas as pd
import requests, json

In [38]:
data = []
count = 3
offset = 0
while count > 0:
    response = requests.get(
        "https://api.restcountries.com/countries/v5",
        headers={"Authorization": f"Bearer rc_live_7758de5a6412468aaf6168642ecc15ee"},
        params={
            "response_fields": (
                "names.common,region,population,area.kilometers,"
                "languages,currencies,borders"
            ),
            "limit": 100,
            "offset": offset,
        },
        timeout=30,
    )
    response.raise_for_status()

    page = response.json()["data"]
    data.extend(page["objects"])

    offset += 100
    count -= 1

In [39]:
with open("raw_data.json", "w") as f:
    json.dump(data, f)

In [40]:
rows = []
for country in data:
    lang = []
    for l in country['languages']:
        lang.append(l["name"])

    curr_codes = []
    for curr in country['currencies']:
        curr_codes.append(curr["code"])

    rows.append({
        "CountryName": country["names"]["common"],
        "Region": country["region"],
        "Area(km2)": country["area"]["kilometers"],
        "Languages": ", ".join(lang),
        "CurrencyCodes": ", ".join(curr_codes),
        "borders": ", ".join(country["borders"]),
        "Population": country["population"],
    })

df = pd.DataFrame(rows)

In [41]:
df

,CountryName,Region,Area(km2),Languages,CurrencyCodes,borders,Population
0,Abkhazia,Asia,8665.0,"Abkhaz, Russian",RUB,"GEO, RUS",244236
1,Afghanistan,Asia,652230.0,"Dari, Pashto, Turkmen",AFN,"IRN, PAK, TKM, UZB, TJK, CHN",43844000
2,Albania,Europe,28748.0,Albanian,ALL,"MNE, GRC, MKD, UNK",2335930
3,Algeria,Africa,2381741.0,Arabic,DZD,"TUN, LBY, NER, ESH, MRT, MLI, MAR",47400000
4,American Samoa,Oceania,199.0,"English, Samoan",USD,,43268
...,...,...,...,...,...,...,...
249,Western Sahara,Africa,266000.0,"Hassaniya Arabic, Spanish","DZD, MAD, MRU","DZA, MRT, MAR",510713
250,Yemen,Asia,527968.0,Arabic,YER,"OMN, SAU",32684503
251,Zambia,Africa,752612.0,English,ZMW,"AGO, BWA, COD, MWI, MOZ, NAM, TZA, ZWE",23003214
252,Zimbabwe,Africa,390757.0,"Barwe, English, Kalanga, Khoisan languages, Nd...","USD, ZWG","BWA, MOZ, ZAF, ZMB",16175966


In [42]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 254 entries, 0 to 253
Data columns (total 7 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   CountryName    254 non-null    object 
 1   Region         254 non-null    object 
 2   Area(km2)      254 non-null    float64
 3   Languages      254 non-null    object 
 4   CurrencyCodes  254 non-null    object 
 5   borders        254 non-null    object 
 6   Population     254 non-null    int64  
dtypes: float64(1), int64(1), object(5)
memory usage: 14.0+ KB


In [49]:
df.isna().sum()

CountryName      0
Region           0
Area(km2)        0
Languages        0
CurrencyCodes    0
borders          0
Population       0
dtype: int64

In [50]:
df[df["borders"] == ""]

,CountryName,Region,Area(km2),Languages,CurrencyCodes,borders,Population
4,American Samoa,Oceania,199.0,"English, Samoan",USD,,43268
7,Anguilla,Americas,91.0,English,XCD,,16010
8,Antarctica,Antarctic,14000000.0,,,,1300
9,Antigua and Barbuda,Americas,442.0,English,XCD,,106365
12,Aruba,Americas,180.0,"Dutch, Papiamento",AWG,,110625
...,...,...,...,...,...,...,...
240,United States Minor Outlying Islands,Americas,34.2,English,USD,,300
241,United States Virgin Islands,Americas,347.0,English,USD,,87146
244,Vanuatu,Oceania,12189.0,"Bislama, English, French",VUV,,321409
248,Wallis and Futuna,Oceania,142.0,French,XPF,,11151


In [53]:
df["Area(km2)"].describe()

count    2.540000e+02
mean     5.917312e+05
std      1.895959e+06
min      4.400000e-01
25%      1.439750e+03
50%      6.297900e+04
75%      3.534238e+05
max      1.709824e+07
Name: Area(km2), dtype: float64

In [54]:
df["Population"].describe()

count    2.540000e+02
mean     3.177917e+07
std      1.314092e+08
min      0.000000e+00
25%      2.220078e+05
50%      4.920816e+06
75%      2.054057e+07
max      1.429404e+09
Name: Population, dtype: float64

In [58]:
df["Region"].value_counts()

Region
Africa       60
Americas     56
Europe       54
Asia         52
Oceania      27
Antarctic     5
Name: count, dtype: int64

In [63]:
desc2 = df["Population"].describe()
Q1 = desc2["25%"]
Q3 = desc2["75%"]
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
num_of_outliers = len(df[(df["Population"] < lower) | (df["Population"] > upper)])

num_of_outliers

30

In [64]:
desc1 = df["Area(km2)"].describe()
Q1 = desc1["25%"]
Q3 = desc1["75%"]
IQR = Q3 - Q1

lower = Q1 - 1.5 * IQR
upper = Q3 + 1.5 * IQR
num_of_outliers = len(df[(df["Area(km2)"] < lower) | (df["Area(km2)"] > upper)])

num_of_outliers

35

In [68]:
len(df[df.duplicated()])

0

In [69]:
df[df["Area(km2)"] < 0]

,CountryName,Region,Area(km2),Languages,CurrencyCodes,borders,Population


In [70]:
df[df["Area(km2)"] == 0]

,CountryName,Region,Area(km2),Languages,CurrencyCodes,borders,Population


In [71]:
df[df["Population"] <= 0]

,CountryName,Region,Area(km2),Languages,CurrencyCodes,borders,Population
29,Bouvet Island,Antarctic,49.0,Norwegian,,,0
97,Heard Island and McDonald Islands,Antarctic,412.0,English,,,0
